# Coverage QA Validation

## 1. Purpose

Coverage asks how completely the generated output represents the material information in the authoritative context. Unsupported additions are intentionally evaluated by faithfulness instead.


## 2. Imports and output location

The helper locates the repository root whether Jupyter starts at the repository root or inside `notebooks/qa`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from idp_eval import EvaluationCase, EvaluationFramework, create_azure_judge
from idp_eval.judges import AzureJudgeConfig

from idp_eval import CoverageEvaluator


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "idp_eval").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the idp-eval repository.")


REPO_ROOT = find_repo_root()
OUTPUT_DIR = REPO_ROOT / "qa_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 3. Configure the judge

Replace every placeholder before running. No Phoenix server is required. If your
application already constructs a compatible judge, you may replace this cell
with that existing construction. Keep credentials in your application's secret
management system rather than saving them in this notebook.


In [ ]:
azure_config = AzureJudgeConfig(
    model="YOUR_AZURE_DEPLOYMENT",
    azure_endpoint="YOUR_AZURE_ENDPOINT",
    tenant_id="YOUR_TENANT_ID",
    client_id="YOUR_CLIENT_ID",
    client_secret="YOUR_CLIENT_SECRET",
    api_version="2024-12-01-preview",
    timeout=180,
    proxy_url=None,
    verify_ssl=True,
    reasoning_effort=None,
)

judge = create_azure_judge(config=azure_config)


## 4. Five mock evaluation cases

These cases intentionally span clear pass, partial, and fail behaviors. `expected_behavior` is a human QA aid, not an exact model-score assertion.


In [ ]:
cases = [
    EvaluationCase(
        case_id="COV-001",
        input="Summarize the service requirements.",
        context={
            "requirements": [
                "Encrypt data at rest.",
                "Retain audit logs for 90 days.",
                "Notify administrators of critical incidents within 15 minutes.",
            ]
        },
        output={
            "security": "Data is encrypted at rest.",
            "audit": "Audit logs are retained for 90 days.",
            "incident_response": "Administrators receive critical-incident alerts within 15 minutes.",
        },
    ),
    EvaluationCase(
        case_id="COV-002",
        input="Summarize the service requirements.",
        context={
            "requirements": [
                "Encrypt data at rest.",
                "Retain audit logs for 90 days.",
                "Notify administrators of critical incidents within 15 minutes.",
            ]
        },
        output={
            "security": "Data is encrypted at rest.",
            "incident_response": "Administrators receive critical-incident alerts within 15 minutes.",
        },
    ),
    EvaluationCase(
        case_id="COV-003",
        input="Describe supported deployment regions.",
        context="The service must support both US and EU deployment regions.",
        output="The service supports deployment in the US region.",
    ),
    EvaluationCase(
        case_id="COV-004",
        input="Summarize the onboarding requirements.",
        context={
            "requirements": [
                "Verify customer identity.",
                "Complete onboarding within 3 business days.",
                "Provide real-time status updates.",
                "Send a completion notification.",
            ]
        },
        output="The workflow verifies customer identity.",
    ),
    EvaluationCase(
        case_id="COV-005",
        input="Describe the operational service contract.",
        context={
            "service_limits": {"requests_per_minute": 1000, "payload_mb": 10},
            "roles": {"administrators": "manage users", "auditors": "read-only access"},
            "availability": "99.9% monthly uptime",
            "notifications": {"critical_incidents": "within 15 minutes"},
        },
        output={
            "limits": "Up to 1,000 requests per minute and 10 MB payloads.",
            "access": "Administrators manage users; auditors have read-only access.",
            "availability": "99.9% monthly uptime.",
            "alerts": "Critical incidents trigger notifications within 15 minutes.",
        },
    ),
]

expected_behavior = {
    "COV-001": "complete coverage",
    "COV-002": "partial coverage — one important requirement omitted",
    "COV-003": "partial coverage — EU qualifier omitted",
    "COV-004": "poor coverage — most source information missing",
    "COV-005": "complete or nearly complete structured coverage",
}


## 5. Inspect the mock inputs


In [ ]:
case_rows = []
for case in cases:
    case_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "input": getattr(case, "input"),
        "context": getattr(case, "context"),
        "output": getattr(case, "output"),
    })

cases_df = pd.DataFrame(case_rows)
display(cases_df)


## 6. Configure one evaluator and Excel output

This notebook runs exactly one metric. `resume=False` creates a fresh QA workbook and no Phoenix tracing is configured.


In [ ]:
excel_path = OUTPUT_DIR / "coverage_validation.xlsx"
evaluator = CoverageEvaluator(max_items=None, reason_mode="overall", verbose=True)
framework = EvaluationFramework(
    evaluators=[evaluator],
    judge=judge,
    output="excel",
    excel_path=str(excel_path),
    resume=False,
)

results = framework.evaluate_many(
    cases,
    run_name="qa-validation",
    dataset_name="mock-acceptance-cases",
    show_progress=True,
)


## 7. Result summary


In [ ]:
METRIC_NAME = "coverage"
summary_rows = []
for case, result_map in zip(cases, results, strict=True):
    result = result_map[METRIC_NAME]
    summary_rows.append({
        "case_id": case.case_id,
        "expected_behavior": expected_behavior[case.case_id],
        "score": result.score,
        "label": result.label,
        "explanation": result.explanation,
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)


## 8. Inspect Excel output

The workbook summary is in `evaluations`; item-level evidence is in `coverage_items`.


In [ ]:
evaluations_df = pd.read_excel(excel_path, sheet_name="evaluations")
display(evaluations_df)

details_df = pd.read_excel(excel_path, sheet_name="coverage_items")
display(details_df)


## 9. Sanity assertions

These assertions validate framework/output behavior and broad direction only; they do not require exact LLM-generated fractions.


In [ ]:
assert len(results) == 5
assert excel_path.exists()
assert all(METRIC_NAME in result_map for result_map in results)
assert len(evaluations_df) == 5
assert set(evaluations_df["key_id"]) == {case.case_id for case in cases}
assert len(details_df) >= 5
assert results[0][METRIC_NAME].score >= results[3][METRIC_NAME].score
print("Coverage QA sanity checks passed.")


## 10. Close judge resources and report the workbook path


In [ ]:
judge.close()
print(f"Excel output: {excel_path.resolve()}")
